# Welcome to dymoval's tutorial!
 
In this tutorial we show the main functionalities of *dymoval*. In particular, we will show:

- How to clean up a measurement dataset,
- How to use such a dataset for validating a model. 

## DC Motor model
Assume that we developed a model of a DC electric motor connected to a constant load.  
It is now time for the truth: can we trust the simulation results?

To figure it out, we need to validate our DC motor model, and *dymoval* is all about that.

But how does our DC motor model look like? 
The model includes interaction between DC voltage (*input*), and the motor speed and the DC current (*outputs*), as shown in the following picture:

<p align="center">
    <br>
  <img src="./DCMotor.svg" width="750" title="DCMotor" class="center">
</p>

### Validation setup
To validate our model, we need to run some experiments on the real DC motor and log some measurements.

We assign the task to a team of engineers, asking them to log measurements in `HDF5` format. 
We ask to log the supply voltage, and the measured armature current and the motor speed, since their interactionn is what is included in our model.
They made us happy and provided us with the requested logs. 

## Log data clean-up
The first step now is to extract the measurements from the `HDF5` and take a look at them. We import some typical packages that suit our need:

In [1]:
%matplotlib inline
import h5py
import numpy as np
import dymoval as dmv

# This is our model
from dymoval_tutorial.DCMotorModel import (
    DCMotor_model_dt,
)
from scipy.signal import dlsim

# set max number of displayed decimals
np.set_printoptions(precision=4, suppress=True)

Next, let's read the log data stored in the `DCMotor_measurements.h5` file. All the logs that we are interested in are stored in the group "*signals*".
Feel free to explore the hdf5 file at your convenience.
We use the `h5py` package to read it, but feel free to explore any other package that can read HDF5 files.

In [2]:
measurements_file = "./DCMotor_measurements.h5"
logs = h5py.File(measurements_file, "r")

We extract the logged signals as it follows (see *h5py* docs for more info):

In [3]:
V = logs["signals/V_measured"]  # Measured supply voltage (input)
Ia = logs["signals/Ia_measured"]  # Measured current (output)
motor_speed = logs[
    "signals/motor_speed_measured"
]  # Measured motor rotational speed (output)

To begin with, we map the measured values into *dymoval* `Signals`.

A dymoval `Signal` is a small dataclass wrapping a 1-D `np.ndarray` along with
the quantities that typically characterize a physical signal:

- `name`,
- `values`,
- `time`,
- `unit`,
- `time_unit`.

The `time` vector is optional: when it is omitted the signal is simply plotted
against the sample index. Here the logger stored a sampling period per signal,
so we can reconstruct the time vector ourselves.

Let's map the logs into dymoval `Signals`:

In [4]:
signal_list = []
for val in [V, Ia, motor_speed]:
    sampling_period = float(val.attrs["period"])
    samples = np.asarray(val[:], dtype=float)
    signal_list.append(
        dmv.Signal(
            name=str(val.attrs["name"]),
            values=samples,
            time=np.arange(len(samples)) * sampling_period,
            unit=str(val.attrs["unit"]),
            time_unit=str(val.attrs["sampling_unit"]),
        )
    )

logs.close()

Let's see how these measurements look like. The signals are not aligned yet. They have different sampling periods and different lengths - so we cannot
put them in a `Dataset` right away. The `plot_signals()` function is there
exactly for this: it plots a bunch of loose `Signals`, one subplot each.

In [5]:
fig = dmv.plot_signals(*signal_list, with_scope=False)

> **A note on the plots.** Every *dymoval* plotting function accepts a
> `with_scope` argument. When it is `True` (the default) the figure comes
> with an interactive side panel that lets you toggle, highlight and
> inspect the plotted signals. That panel only makes sense with an
> interactive Matplotlib backend, e.g. `%matplotlib qt` or
> `%matplotlib widget`. Since this notebook uses the static `inline`
> backend, we pass `with_scope=False` throughout. Feel free to switch
> backend and drop that argument to play with the scopes.

The first thing we notice is that signals have been logged with different sampling period. There is also some missing data and the overall measurements look noisy.

Well, things happen in the lab! And to make things even worst, it does not seem possible to run more experiments before the next month.

We better off to clean these logs. A good measurements dataset should have all the signals sampled with the same sampling period, it should not contains missing data and the signal-to-noise ratio should be sufficiently high.

Firstly, we see that in the interval about $T = [40,70]\,s$ all the signals have no missing data. We can take that portion to validate our model.
However, the signals have been sampled with different sampling periods. 

Fortunately, *dymoval* allows us to trim signals through `Signal.trim()` and to
re-sample them when they are gathered into a `Dataset`. Let's trim first:

In [6]:
signal_list = [s.trim(40.0, 70.0) for s in signal_list]

# No missing data left, but still two different sampling periods
[(s.name, len(s), s.get_sampling_period()) for s in signal_list]

[('Supply_Voltage', 3001, 0.01),
 ('Armature_Current', 3001, 0.01),
 ('Motor_Speed', 6001, 0.005)]

Now we can build a `Dataset`, i.e. a set of *aligned* signals split into
inputs and outputs. The `Dataset.from_signals()` factory takes care of the
alignment for us: signals that do not already share the same time vector are
re-sampled on a common, uniform grid.

By default the grid uses the *largest* sampling period found among the signals
(so that no data is invented) and spans the time interval covered by all of
them. We can also choose the grid explicitly through
`target_sampling_period`, which is what we do here:

In [7]:
u_signals = signal_list[:1]  # Supply voltage
y_signals = signal_list[1:]  # Armature current and motor speed

ds = dmv.Dataset.from_signals(
    inputs=u_signals,
    outputs=y_signals,
    target_sampling_period=0.01,
    meta={"name": "DCMotor"},
)
ds

Dataset: 1 input(s), 2 output(s), 3001 samples, dt = 0.01 s
  INPUT  Supply_Voltage [V]
  OUTPUT Armature_Current [A]
  OUTPUT Motor_Speed [RPM]

Note that the strict `Dataset(...)` constructor is still available, but it
*requires* the signals to be already aligned - it will not silently re-sample
anything. Use the `from_signals()` / `from_dict()` factories whenever the
alignment has to be done for you.

Let's now take a look at our cleaned `Dataset`:

In [8]:
fig = ds.plot(with_scope=False)

The dataset is not bad, but the measurements are a bit noisy. Let's clean them up a little bit through a low pass filter. But wait! How to choose the cutoff frequency?

Well, we can perform a spectral analysis.
We plot the spectrum through the method `plot_spectrum()` but before we remove the continuous component through the `remove_mean()` method given that the signals have a fairly large continuous component:

In [9]:
fig = ds.remove_mean().plot_spectrum(with_scope=False)

The signals bandwidth of two over three signals is in contained very a narrow range of about $f < 2\, Hz$.

To avoid losing transient information and worsening the residuals analysis that we will do later on, we stay a bit larger than $2\, Hz$. We choose a cutoff frequency $f_c = 5\, Hz$.

In [10]:
cutoff = 5  # [Hz]
ds_filt = ds.low_pass_filter(
    ("Supply_Voltage", cutoff),
    ("Motor_Speed", cutoff),
    ("Armature_Current", cutoff),
)

As you see from the function call, you could use different cutoff frequencies for each signal.

Note that every processing method of `Dataset` (`trim()`, `remove_mean()`,
`low_pass_filter()`, `apply()`, ...) is **immutable**: it returns a brand new
`Dataset` and leaves the original one untouched. This is why we can keep
comparing `ds` and `ds_filt` below.

Let's take a look at the low-pass-filtered measurements:

In [11]:
fig = ds_filt.plot(with_scope=False)

...and let's compare our datasets before and after the filtering:

In [12]:
fig = dmv.plot_compare(
    ds, ds_filt, labels=["raw", "filtered"], with_scope=False
)

The chosen cutoff frequency seems to clean up most of the noise and at the same time it does not intoduce too much lag.

Let's see if we cut too much signal from the spectrum plot:

In [13]:
fig = dmv.plot_spectrum_compare(
    ds.remove_mean(),
    ds_filt.remove_mean(),
    labels=["raw", "filtered"],
    mode="power",
    with_scope=False,
)

It looks like that we just cut the noise, but most of the signal is still there.

Next, let's check the coverage of our filtered dataset.

In [14]:
fig = ds_filt.plot_coverage()

Knowing the coverage is extremely important and it shall always be delivered along with the model. The coverage region tell you the region where you can trust your model, simply because... well, you are going to validate your model in such a region.

Classic statistics such as means and covariances of the various signals are available programmatically:

In [15]:
u_mean, u_cov, y_mean, y_cov = ds_filt.coverage()
print("input mean:\n", u_mean)
print("input covariance:\n", u_cov)
print("output mean:\n", y_mean)
print("output covariance:\n", y_cov)

input mean:
 [10.1589]
input covariance:
 [[0.4237]]
output mean:
 [  0.1008 960.3171]
output covariance:
 [[   0.0001    0.3679]
 [   0.3679 3699.7314]]


Next, only for the sake of showing more dymoval features, we can selectively choose the signals that we want to plot or we can overlap them if we pass them as a tuple:

In [16]:
fig = ds_filt.plot(
    ("Armature_Current", "Supply_Voltage"), "Motor_Speed", with_scope=False
)

## Assess input excitation

Before validating a model, let's check the input independently of any model. The lagged-input Gram matrix combines input channels and their delayed samples. Its smallest eigenvalue describes the weakest excited combination.

Here we inspect fluctuations around the recorded mean voltage, rather than counting the DC operating level as excitation. We use the unfiltered dataset so this check describes the recorded input rather than the smoothing introduced by our filter. Outputs are not used in this computation.

We choose `L = 20` samples as an illustration. At our sampling period of 0.01 s, the oldest and newest samples in each stack are 0.19 s apart. In your own experiment, choose a length relevant to the dynamics you want to validate.

In [17]:
L = 20
ds_excitation = ds.remove_mean()
input_eigenvalues = ds_excitation.input_gram_eigenvalues(window_length=L)
print(f"Minimum input Gram eigenvalue: {input_eigenvalues[0]:.6g}")
print(f"Maximum input Gram eigenvalue: {input_eigenvalues[-1]:.6g}")

# If you need the averaged matrix itself:
input_gram = ds_excitation.input_gram_matrix(window_length=L)

Minimum input Gram eigenvalue: 0.0476401
Maximum input Gram eigenvalue: 9.69858


A near-zero minimum eigenvalue suggests a weakly excited combination. There is no universal pass/fail threshold: the values depend on `L`, input amplitudes, and channel units. The methods do not automatically center or scale inputs; we explicitly removed the means above.

This is an input-excitation check, not a certificate of dataset suitability. Inspect the input/output plots too, checking that the outputs respond clearly above measurement noise and that the experiment covers the relevant frequencies and operating conditions. These dataset diagnostics are separate from the model-validation report.

## Validation Session

Here is what it is our table so far:

- cleaned-up measurements stored in a dymoval *Dataset* object,
- DC motor model.

It is now time for validation! 

The validation consists in comparing the *measured out* (cleaned and stored in out *Dataset* object `ds_filt`) with the *simulated outputs* from our model (to be obtained) when it is stimulated with the *measured input*.

Therefore, let's extract the input `u` from the measurements `Dataset` and let's feed our model with it: 

In [18]:
(t, u, y) = ds_filt.dataset_values()

# Generated simulated out
_, y_sim, _ = dlsim(DCMotor_model_dt, u=u, x0=[0.0, 0.0, 0.0])

We use the `validate_models()` function to validate our model. Such a function takes measured in/out data along with the simulated out, and it returns a `ValidationSession` object that contains all the information about the validation outcomes.

The measured in/out can be passed either as a `np.ndarray` or as a list of dymoval `Signal`. We opt for the latter option, which we get out of the `Dataset` through `to_signals()`. The simulated outputs are passed as a `np.ndarray`:

In [19]:
measured_signals = ds_filt.to_signals()
measured_in = measured_signals["INPUT"]
measured_out = measured_signals["OUTPUT"]

vs = dmv.validate_models(
    measured_in=measured_in,
    measured_out=measured_out,
    simulated_out=y_sim,
    Ruu_envelope="auto",
    Ruu_nlags = np.array([[100]]).T,
    Ree_nlags = np.array([[100, 100], [100, 100]]),
    Rue_nlags = np.array([[100], [100]]).T,
)
vs

Validation session name: quick & dirty

Validation setup:
----------------
Inputs auto-correlation
Statistic: abs_mean-max
Ruu_local_weights: None
Ruu_global_weights: None
num lags: 100
Gaussian limits (b0, b_inf, sample_scale): {'Supply_Voltage': (0.95, 0.1, 5.0)}
Envelope ratio <= threshold passes; zero lag excluded.

Residuals auto-correlation:
Statistic: abs_mean-max
Ree_local_weights: None
Ree_global_weights: None
num lags: 100
Gaussian limits (b0, b_inf, sample_scale): {'Armature_Current': (0.95, 0.1, 5.0), 'Motor_Speed': (0.95, 0.1, 5.0)}
Envelope ratio <= threshold passes; zero lag excluded.

Input-residuals cross-correlation:
Statistic: abs_mean-max
Rue_local_weights: None
Rue_global_weights: None
num lags: 100

Validation results:
-------------------
Thresholds: 
r2: 35.0000 
Rue_correlation: 0.5000 
Ruu_envelope: 1.0000 
Ree_envelope: 1.0000 
Ree_cross_correlation: 0.5000 

Actuals:
                                                            Sim_0
Input cross-correlation (ab

As we can see, our model failed the test. In particular, the value or the R-squared index is very bad. Let's take a look at the simulated VS measured out plot.

In [20]:
fig = vs.plot_simulations(dataset="out", with_scope=False)

As you can see, there is a large mismatch around `t = 0` between the measured and the simulated out, and that may be the reason of such a poor r-squared value. 

Such a mismatch is due to that around `t=0` the real DC Motor was in steady state, whereas our model started at initial condition `[0.0, 0.0 ,0.0]`, and therefore we were comparing "apples with oranges" in such a region. To fix this, either we adjust the initial condition of our model to match the measurement data, or we just neglect that area in the validation. We go for the latter solution by trimming the `ValidationSession` time span.

In [21]:
vs_trimmed = vs.trim(1, 30)
fig = vs_trimmed.plot_simulations(dataset="out", with_scope=False)

The plots look much better. Let's take a look at the validation outcome:

In [22]:
vs_trimmed

Validation session name: quick & dirty

Validation setup:
----------------
Inputs auto-correlation
Statistic: abs_mean-max
Ruu_local_weights: None
Ruu_global_weights: None
num lags: 100
Gaussian limits (b0, b_inf, sample_scale): {'Supply_Voltage': (0.95, 0.1, 5.0)}
Envelope ratio <= threshold passes; zero lag excluded.

Residuals auto-correlation:
Statistic: abs_mean-max
Ree_local_weights: None
Ree_global_weights: None
num lags: 100
Gaussian limits (b0, b_inf, sample_scale): {'Armature_Current': (0.95, 0.1, 5.0), 'Motor_Speed': (0.95, 0.1, 5.0)}
Envelope ratio <= threshold passes; zero lag excluded.

Input-residuals cross-correlation:
Statistic: abs_mean-max
Rue_local_weights: None
Rue_global_weights: None
num lags: 100

Validation results:
-------------------
Thresholds: 
r2: 35.0000 
Rue_correlation: 0.5000 
Ruu_envelope: 1.0000 
Ree_envelope: 1.0000 
Ree_cross_correlation: 0.5000 

Actuals:
                                                            Sim_0
Input cross-correlation (ab

The fit improves after trimming, but the overall result is still FAIL: the input-correlation score exceeds its default threshold. Correlations now use every sample lag, so the slowly varying input has strong short-delay correlation. This is an excitation diagnostic, not proof that the model is bad. Inspect input richness and output responses before deciding whether this experiment supports your intended validation.

The outcome is also available as a plain dictionary, which is handy if you want
to assert on it in a CI pipeline:

In [23]:
print(vs_trimmed.outcome)
# Residual RMSE is included below (max across outputs by default).
# Outputs with different units should be scaled before aggregating RMSE.
# No RMSE threshold is enabled by default.
print(vs_trimmed.validation_statistics)

{'Sim_0': 'FAIL'}
{'Sim_0': {'Ruu_correlation': 0.8449090885967412, 'r2': 65.90086683495915, 'rmse': 25.10898551737754, 'Ree_correlation': 0.32787353822359455, 'Rue_correlation': 0.22555959640692055, 'Ruu_envelope': 9.560562806357945, 'Ree_cross_correlation': 0.0933471268507492, 'Ree_envelope': 5.190941759563196}}


Finally, the residuals analysis behind those numbers can be inspected
graphically. `plot_residuals()` returns the input auto-correlation
$R_{uu}$, the residuals auto-correlation $R_{ee}$ and the input-residuals
cross-correlation $R_{ue}$. Every lag is one sampling period. The diagonal of $R_{ee}$ contains individual residual auto-correlations; its off-diagonal entries and all entries of $R_{ue}$ are cross-correlations. Only diagonal self-correlation peaks at zero are excluded from scoring. Low correlations are useful evidence, not proof that all dynamics are captured.

In [24]:
figs = vs_trimmed.plot_residuals()

The output comparisons and residuals provide evidence about the model within the recorded coverage region, but the failed input check still needs engineering assessment. The results are not a guarantee of model suitability.

### Customize Gaussian input and residual limits

You can assess diagonal input and residual auto-correlations against separate Gaussian acceptance envelopes. Each tuple is `(short_delay_limit, long_delay_floor, time_scale)`; time scales here are in seconds. These values are illustrative, not calibrated acceptance limits. Use application requirements or a trusted reference model evaluated on representative logs, then freeze the limits before testing candidates.

`Ruu_envelope` and `Ree_envelope` report the maximum nonzero-lag ratio across input and output diagonals. Each replaces its ordinary diagonal check in default thresholds and the report row. Ordinary diagonal scores remain under `Ruu_correlation` and `Ree_correlation`; `Ruu_cross_correlation` and `Ree_cross_correlation` separately score off-diagonal pairs. The plots shade the Gaussian limits on diagonals only. Input-residual scoring is unchanged.

In [25]:
input_limits = {vs_trimmed.dataset.input_names()[0]: (0.95, 0.1, 0.05)}
limits = {
    "Armature_Current": (0.95, 0.1, 0.05),
    "Motor_Speed": (0.95, 0.1, 0.2),
}
vs_envelope = dmv.ValidationSession(
    "Illustrative residual limits", vs_trimmed.dataset,
    Ruu_envelope=input_limits,
    Ree_envelope=limits,
)
vs_envelope = vs_envelope.append_simulation(
    "Sim_0", vs_trimmed.dataset.output_names(),
    np.column_stack([s.values for s in vs_trimmed.simulations["Sim_0"]]),
)
print(vs_envelope)
envelope_figures = vs_envelope.plot_residuals(plot_input=False)

Validation session name: Illustrative residual limits

Validation setup:
----------------
Inputs auto-correlation
Statistic: abs_mean-max
Ruu_local_weights: None
Ruu_global_weights: None
num lags: 41
Gaussian limits (b0, b_inf, sample_scale): {'Supply_Voltage': (0.95, 0.1, 5.0)}
Envelope ratio <= threshold passes; zero lag excluded.

Residuals auto-correlation:
Statistic: abs_mean-max
Ree_local_weights: None
Ree_global_weights: None
num lags: 41
Gaussian limits (b0, b_inf, sample_scale): {'Armature_Current': (0.95, 0.1, 5.0), 'Motor_Speed': (0.95, 0.1, 20.0)}
Envelope ratio <= threshold passes; zero lag excluded.

Input-residuals cross-correlation:
Statistic: abs_mean-max
Rue_local_weights: None
Rue_global_weights: None
num lags: 41

Validation results:
-------------------
Thresholds: 
r2: 35.0000 
Rue_correlation: 0.5000 
Ruu_envelope: 1.0000 
Ree_envelope: 1.0000 
Ree_cross_correlation: 0.5000 

Actuals:
                                                            Sim_0
Input cross-co

Gaussian scoring is enabled by default: `Ruu_envelope="auto"` and `Ree_envelope="auto"` each use `(0.95, 0.1, 5*Ts)` as starting tolerances, not calibrated limits. The example above customizes them independently. Inspect both sides of the shaded regions; zero lag is plotted but not scored on diagonals. Increasing the short-delay limit or time scale relaxes an envelope; lowering the floor tightens its long-delay requirement. Do not adjust these values merely to obtain PASS. Keep evaluation windows fixed across comparisons. Set either envelope to `None` for ordinary diagonal scoring. If you supply custom thresholds, explicitly include `"Ruu_envelope": 1.0` and/or `"Ree_envelope": 1.0` to check enabled envelopes.

## Conclusions

Let's make a small summary and rconnect to the theoretical results:

- Output plots: the simulated and measured outputs look very similar. This is also reflected by the high $R^2$ value. However, for control system design, the most important aspect to check are the "peaks & dips" of the simulated response happen correspondently to the measured value, rather than the exact values. This because most likely your control algorithm has an integral action that would compensate for such a mismatch,
- R-squared and residual RMSE describe fit and error magnitude. Acceptable values depend on the application; there is no universal fit target. The tutorial outputs have different units, so the aggregated RMSE requires care. Use `dmv.rmse(y_measured, y_simulated)` to inspect per-output errors separately.
- Input correlation $R_{uu}$: the slowly varying input fails the default correlation threshold on the full-resolution lag grid. Assess its excitation separately using plots and Gram-matrix eigenvalues; low correlation alone would not establish dataset suitability.
- Residual correlation $R_{ee}$ reveals temporal or shared error patterns. A high short-delay peak can come from model mismatch, colored noise, or disturbances; a slowly varying output does not automatically imply slowly varying residuals. Default Gaussian limits are starting tolerances; customize them for your experiment rather than treating them as a universal test.
- Input-residual cross-correlation $R_{ue}$ checks whether errors relate to the applied input. Inspect peaks as well as aggregate values: `abs_mean` depends on the window and can hide localized peaks.
- Coverage region: these measurements cover roughly 9.0--11.5 Volts, 0.08--0.12 Ampere, and 850--1100 RPM. Interpret the validation evidence within these conditions, without assuming that it extends outside them.

In general, when validating models, we are more interested in looking at "shapes" than exact numbers: we visually inspect "dips & peaks" of the outputs, the shape of the various correlation functions and so on, and we use engineering judgement to promote or reject a model. Nevertheless, in this case, we also have a pretty high numerical accuracy.

However, in a CI setting, visual inspections are difficult. Therefore, we use compare some evaluation metrics to some thresholds. Given the MIMO nature of the system, dymoval evaluate the whiteness levels by first computing some norm of signal of each constituent scalar signal; once done, it computes some norm of the obtained matrix of scalar values. In this tutorial, we first compute the mean value of each signal, and then we take the absolute max value or the obtained matrix (abs_mean-max metric, see validation results) and we compare them to some thresholds. 

We finally wish to remark that what we did so far is independent of the modeling language used. In-fact *dymoval* only looks at the output generated by the model, not how the modeling has been performed or what modeling tool has been used. 
